# Control de acceso con reconocimiento facial: de scripts sueltos a un pipeline reutilizable

- Sistema de control de acceso que identifica personas por rostro en video, sin gafetes ni claves.
- El origen fue una serie de notebooks independientes: grabar video, extraer rostros, entrenar el modelo y reconocer en tiempo real vivían cada uno por separado, con rutas de una sola máquina.
- Este cuaderno consolida ese flujo en un pipeline único, reproducible de principio a fin.
- **Nota de transparencia:** las identidades reales del proyecto original se sustituyen aquí por el dataset público *AT&T/Olivetti Faces*, más metadata sintética (nombre, cargo, eventos de entrada/salida). Ningún dato de este cuaderno corresponde a una persona real del proyecto original.
- Artículo completo, con el contexto de ingeniería y las decisiones detrás de cada paso: [https://fuzzyfrog.ai/es/ai-lab/proyectos/industria/reconocimiento-facial-control-acceso-opencv-mediapipe/](https://fuzzyfrog.ai/es/ai-lab/proyectos/industria/reconocimiento-facial-control-acceso-opencv-mediapipe/)


## Diagrama del pipeline

```
📹 Frame de video
        │
        ▼
🧭 Detección de rostro (MediaPipe Face Detection)
        │  recorte + normalización
        ▼
🗂️ Dataset de rostros + metadata (identidad, cargo)
        │
        ▼
🤖 Entrenamiento del reconocedor (LBPH)
        │
        ▼
🔍 Inferencia en tiempo real + umbral de confianza
        │
        ▼
🚪 Registro de entrada/salida (o rechazo si no hay match)
```

Los notebooks originales resolvían cada bloque como un script aislado. Aquí se mantiene el mismo criterio técnico, pero encadenado como funciones que se pueden importar y reusar.


## Carga de datos

- Usamos **AT&T/Olivetti Faces**: 40 identidades, 10 fotos por identidad, 64x64 en escala de grises. Es un dataset de dominio público, construido específicamente para investigación en reconocimiento facial.
- Sirve como reemplazo estructural de las identidades reales del proyecto original: mismo tipo de dato (rostro recortado en escala de grises), mismo comportamiento de clase (pocas fotos por identidad), sin comprometer a nadie.
- Sobre esas 40 identidades generamos metadata sintética: nombre ficticio, cargo, y una bitácora simulada de eventos de entrada/salida, para conservar la lógica de registro que ya tenían los notebooks originales (`Cargo`, `Edad`/`Cedula`, `entrar_salir`).


In [ ]:
# Dependencias puntuales de este cuaderno (Colab ya trae la mayoría)
# !pip install -q opencv-contrib-python-headless "mediapipe==0.10.14" scikit-learn pandas matplotlib


In [ ]:
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import random

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
random.seed(RANDOM_STATE)

# --- Rostros: dataset público (reemplaza identidades reales) ---
faces = fetch_olivetti_faces()
X_faces = faces.images          # (400, 64, 64) float32 en [0, 1]
y_identity = faces.target       # (400,) -> 40 identidades, 10 fotos c/u

print("Imágenes:", X_faces.shape, "| Identidades únicas:", len(set(y_identity)))


In [ ]:
# --- Metadata sintética: nombre, cargo, bitácora de entrada/salida ---
NOMBRES_FICTICIOS = [
    "Ana Torres", "Luis Medina", "Karla Ruiz", "Diego Pardo", "Sofía Nava",
    "Marco Reyes", "Elena Cano", "Iván Soto", "Paula Vega", "Hugo Marín",
    "Renata Cruz", "Tomás Ibarra", "Lucía Prado", "Emilio Rios", "Valeria Ponce",
    "Adrián León", "Camila Duarte", "Sergio Mota", "Fernanda Ochoa", "Raúl Peña",
    "Ximena Solís", "Bruno Castañeda", "Daniela Aguirre", "Gael Montes", "Irene Salas",
    "Mateo Fuentes", "Julia Ríos", "Andrés Bravo", "Regina Campos", "Óscar Lira",
    "Natalia Herrera", "Pablo Cordero", "Alma Guzmán", "Rodrigo Nieto", "Paulina Escobar",
    "Cristian Rendón", "Mariana Zapata", "Leonardo Trejo", "Gabriela Ontiveros", "Iker Delgado",
]
CARGOS = ["Operador de planta", "Supervisor de turno", "Técnico de mantenimiento", "Visitante autorizado"]

metadata = pd.DataFrame({
    "identity_id": range(40),
    "nombre": NOMBRES_FICTICIOS,
    "cargo": [random.choice(CARGOS) for _ in range(40)],
})

# Bitácora simulada de accesos: varios eventos por identidad + algunos intentos "desconocidos"
eventos = []
base_ts = pd.Timestamp("2026-03-02 06:00:00")
for i in range(250):
    es_desconocido = rng.random() < 0.08  # ~8% de intentos sin match, simulan intrusos/no registrados
    identity_id = -1 if es_desconocido else int(rng.integers(0, 40))
    tipo = rng.choice(["entrada", "salida"])
    ts = base_ts + pd.Timedelta(minutes=int(rng.integers(0, 60 * 24 * 10)))
    eventos.append({"evento_id": i, "identity_id": identity_id, "tipo": tipo, "timestamp": ts})

bitacora = pd.DataFrame(eventos).sort_values("timestamp").reset_index(drop=True)

metadata.to_csv("outputs/identidades_sinteticas.csv", index=False)
bitacora.to_csv("outputs/bitacora_accesos_sintetica.csv", index=False)

metadata.head()


## Explicación de los datos

- `X_faces`: 400 imágenes (40 identidades x 10 tomas), 64x64, escala de grises, ya recortadas y alineadas al rostro. En producción esa alineación la hace el paso de detección (MediaPipe), no viene dada.
- `y_identity`: la identidad (0 a 39) de cada imagen. Es el target del reconocedor.
- `metadata`: una fila por identidad con nombre y cargo, análoga a la tabla de personas registradas del sistema original.
- `bitacora`: eventos de entrada/salida simulados, incluyendo un ~8% de intentos sin match (`identity_id = -1`), que representan a alguien que la cámara ve pero el sistema no reconoce.


In [ ]:
print("Forma de X_faces:", X_faces.shape, X_faces.dtype)
print("Rango de intensidad:", X_faces.min(), "-", X_faces.max())
print("\nDistribución de fotos por identidad (debe ser 10 parejo):")
print(pd.Series(y_identity).value_counts().sort_index().describe())
print("\nEjemplo de bitácora:")
bitacora.head()


## Análisis exploratorio

- Revisamos visualmente una muestra de rostros para confirmar que la resolución (64x64) alcanza para que el ojo humano distinga identidades. Es el mismo límite que enfrenta el reconocedor.
- Revisamos la distribución de cargos, para no entrenar y evaluar sobre una bitácora artificialmente pareja.
- Confirmamos que no hay identidades con menos de 2 fotos, condición mínima para poder separar entrenamiento y prueba por identidad.


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    idx = i * 5  # una muestra espaciada entre identidades distintas
    ax.imshow(X_faces[idx], cmap="gray")
    ax.set_title(f"id {y_identity[idx]}", fontsize=8)
    ax.axis("off")
plt.suptitle("Muestra de rostros (dataset público, sustituye identidades reales)")
plt.tight_layout()
plt.show()


In [ ]:
metadata["cargo"].value_counts().plot(kind="bar", figsize=(6, 3), color="#006a87")
plt.title("Distribución de cargos en la bitácora sintética")
plt.ylabel("personas")
plt.tight_layout()
plt.show()

fotos_por_identidad = pd.Series(y_identity).value_counts()
print("Identidades con menos de 2 fotos:", (fotos_por_identidad < 2).sum())


## Modelado

Dos piezas separadas, igual que en los notebooks originales, pero ahora como funciones:

- **Detección** (MediaPipe Face Detection): localiza el rostro dentro de un frame completo de cámara. Aquí se demuestra sobre un frame simulado, porque el dataset de identidades ya viene recortado.
- **Reconocimiento** (LBPH, `cv2.face.LBPHFaceRecognizer_create`): se entrena sobre los rostros ya recortados.

**Decisión de ingeniería que se conserva del proyecto original:** LBPH y no una red profunda de embeddings faciales. Con 10 fotos por identidad, una CNN entrenada desde cero sobrejustaría de inmediato, y un modelo preentrenado de embeddings agrega una dependencia pesada para un caso de uso que corre en una sola cámara y CPU modesta. LBPH es liviano, interpretable y suficiente para el volumen de identidades de una planta pequeña. El costo es que no escala igual de bien a cientos de identidades, algo que sí valdría la pena revisar si el sistema creciera.


In [ ]:
def detectar_rostro(frame_rgb, min_confidence=0.5):
    """Localiza el rostro principal en un frame de cámara (formato RGB).
    Devuelve el recorte en escala de grises 64x64, listo para el reconocedor,
    o None si no se detectó ningún rostro.
    """
    mp_face_detection = mp.solutions.face_detection
    with mp_face_detection.FaceDetection(min_detection_confidence=min_confidence,
                                          model_selection=1) as face_detection:
        resultado = face_detection.process(frame_rgb)
        if not resultado.detections:
            return None

        deteccion = resultado.detections[0]
        h, w, _ = frame_rgb.shape
        bbox = deteccion.location_data.relative_bounding_box
        x1 = max(int(bbox.xmin * w), 0)
        y1 = max(int(bbox.ymin * h), 0)
        x2 = min(int((bbox.xmin + bbox.width) * w), w)
        y2 = min(int((bbox.ymin + bbox.height) * h), h)

        recorte = frame_rgb[y1:y2, x1:x2]
        if recorte.size == 0:
            return None
        recorte_gris = cv2.cvtColor(recorte, cv2.COLOR_RGB2GRAY)
        return cv2.resize(recorte_gris, (64, 64))


# Demo sobre un frame simulado: una imagen del dataset "pegada" sobre un canvas más
# grande, para imitar un frame de cámara real donde el rostro no ocupa todo el cuadro.
frame_demo = np.zeros((240, 320, 3), dtype="uint8")
rostro_ejemplo = (X_faces[0] * 255).astype("uint8")
rostro_ejemplo_rgb = cv2.cvtColor(cv2.resize(rostro_ejemplo, (100, 100)), cv2.COLOR_GRAY2RGB)
frame_demo[70:170, 110:210] = rostro_ejemplo_rgb

recorte = detectar_rostro(frame_demo, min_confidence=0.3)
print("¿Se detectó un rostro en el frame simulado?", recorte is not None)


In [ ]:
# --- Entrenamiento del reconocedor sobre los rostros ya recortados ---
imagenes_uint8 = [(img * 255).astype("uint8") for img in X_faces]

X_train_idx, X_test_idx = train_test_split(
    np.arange(len(imagenes_uint8)),
    test_size=0.2,
    stratify=y_identity,
    random_state=RANDOM_STATE,
)

imgs_train = [imagenes_uint8[i] for i in X_train_idx]
labels_train = y_identity[X_train_idx]
imgs_test = [imagenes_uint8[i] for i in X_test_idx]
labels_test = y_identity[X_test_idx]

recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.train(imgs_train, labels_train)

print(f"Entrenado con {len(imgs_train)} rostros, evaluado con {len(imgs_test)}.")


## Evaluación

- Precisión del reconocedor sobre el conjunto de prueba.
- Un control de acceso real no solo necesita acertar la identidad: necesita saber cuándo **no** reconocer a nadie. LBPH devuelve una distancia (`confidence`, entre más baja mejor), así que definimos un umbral para declarar "desconocido" en vez de forzar el match más parecido.
- Simulamos el registro de entrada/salida aplicando ese umbral, igual que haría el sistema en producción.


In [ ]:
UMBRAL_DESCONOCIDO = 70.0  # distancia LBPH; ajustar con datos reales de la cámara final

aciertos = 0
resultados_eval = []
for img, verdadero in zip(imgs_test, labels_test):
    pred_id, distancia = recognizer.predict(img)
    if distancia > UMBRAL_DESCONOCIDO:
        pred_final = -1  # desconocido, se rechaza el acceso
    else:
        pred_final = pred_id
    resultados_eval.append({"real": verdadero, "predicho": pred_final, "distancia": distancia})
    if pred_final == verdadero:
        aciertos += 1

df_eval = pd.DataFrame(resultados_eval)
precision = aciertos / len(labels_test)
print(f"Precisión con umbral aplicado: {precision:.2%}")
print(f"Casos marcados como 'desconocido' pese a ser identidad válida: {(df_eval['predicho'] == -1).sum()} de {len(df_eval)}")


In [ ]:
# Matriz de confusión simplificada: correcto / incorrecto / rechazado como desconocido
df_eval["resultado"] = np.where(
    df_eval["predicho"] == df_eval["real"], "correcto",
    np.where(df_eval["predicho"] == -1, "rechazado (desconocido)", "incorrecto")
)
print(df_eval["resultado"].value_counts())

df_eval["resultado"].value_counts().plot(kind="bar", figsize=(5, 3), color=["#00b76c", "#e0a800", "#c0392b"])
plt.title("Resultado de la inferencia sobre el set de prueba")
plt.tight_layout()
plt.show()


In [ ]:
# --- Simulación de registro de entrada/salida usando el umbral definido arriba ---
def registrar_evento(pred_id, distancia, tipo, metadata_df, umbral=UMBRAL_DESCONOCIDO):
    if distancia > umbral or pred_id == -1:
        return {"identity_id": -1, "nombre": "DESCONOCIDO", "tipo": tipo, "acceso": "rechazado"}
    nombre = metadata_df.loc[metadata_df["identity_id"] == pred_id, "nombre"].values[0]
    return {"identity_id": pred_id, "nombre": nombre, "tipo": tipo, "acceso": "concedido"}

# Ejemplo con tres rostros del set de prueba
for img, verdadero in list(zip(imgs_test, labels_test))[:3]:
    pred_id, distancia = recognizer.predict(img)
    evento = registrar_evento(pred_id, distancia, tipo="entrada", metadata_df=metadata)
    print(evento)


## Hallazgos principales

- **De prototipo a módulo reutilizable.** El cambio de mayor impacto no fue el modelo, fue la refactorización: pasar de seis notebooks con rutas fijas de una sola máquina a dos funciones (`detectar_rostro`, `registrar_evento`) que cualquiera puede importar. Eso es lo que separa un proyecto que se queda en el portafolio de uno que se puede desplegar.
- **El umbral de "desconocido" importa más que la precisión bruta.** Un sistema de control de acceso que nunca dice "no sé quién eres" es más peligroso que uno un poco menos preciso. Ajustar `UMBRAL_DESCONOCIDO` con datos reales de la cámara final es el paso de calibración más importante antes de producción, no un detalle menor.
- **LBPH fue una decisión consciente, no un default.** Con pocas fotos por identidad y hardware modesto, un modelo profundo de embeddings faciales habría sido sobreingeniería. La ganancia de precisión no compensaba el costo de dependencia y latencia para este caso.
- **Nota de transparencia:** este cuaderno usa el dataset público *AT&T/Olivetti Faces* más metadata sintética (nombres, cargos, bitácora de accesos) en lugar de las identidades reales del proyecto original, manteniendo la misma estructura y comportamiento de los datos.
- Si quieres ver un enfoque publicado reciente sobre reconocimiento facial ligero para control de acceso en entornos con recursos limitados, este es un buen punto de referencia del nivel técnico: [Real-time face recognition based access control system (IEEE)](https://ieeexplore.ieee.org/document/10396794).

Artículo completo, con el contexto de negocio y las decisiones de diseño detrás de cada paso: [https://fuzzyfrog.ai/es/ai-lab/proyectos/industria/reconocimiento-facial-control-acceso-opencv-mediapipe/](https://fuzzyfrog.ai/es/ai-lab/proyectos/industria/reconocimiento-facial-control-acceso-opencv-mediapipe/)

*Hecho con ❤️ por FuzzyFrog.AI*
